**FEATURE SELECTION CODE**

In [0]:
# Databricks only. Run this once if packages are missing.
%pip install -U lightgbm pyarrow pandas numpy scikit-learn psutil

# After install in Databricks:
dbutils.library.restartPython()

In [0]:
import os
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import lightgbm as lgb
import psutil

from sklearn.metrics import r2_score

In [0]:
# ============================================================
# Input / output config
# ============================================================

from pathlib import Path

import numpy as np
import psutil
import pyarrow.dataset as ds

# ============================================================
# Paths
# ============================================================

DATA_PATH = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_parquet"
)

OUTPUT_DIR = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ============================================================
# Column config
# ============================================================

TARGET_COL = "TOTAL_NBRX_C4"

EXCLUDE_COLS = {
    TARGET_COL,
    "PROVIDER_ID"
}

# ============================================================
# Train/test config
# ============================================================

TRAIN_RATIO = 0.80
SEED = 42

# ============================================================
# LightGBM config
# ============================================================

TOP_K = 1000

NUM_ITERATIONS = 250
LEARNING_RATE = 0.05
NUM_LEAVES = 31
MIN_DATA_IN_LEAF = 100

FEATURE_FRACTION = 1.0
BAGGING_FRACTION = 1.0
BAGGING_FREQ = 0

MAX_BIN = 127
BIN_CONSTRUCT_SAMPLE_CNT = 200000

# ============================================================
# Thread config
# ============================================================

NUM_THREADS = psutil.cpu_count(logical=False)

if NUM_THREADS is None:
    NUM_THREADS = 64

NUM_THREADS = min(NUM_THREADS, 64)

# ============================================================
# Data loading config
# ============================================================

ARROW_BATCH_SIZE = 2048

ARRAY_DTYPE = np.float32

# ============================================================
# SHAP config
# ============================================================

SHAP_BATCH_SIZE = 512

# ============================================================
# Quick checks
# ============================================================

print("DATA_PATH:")
print(DATA_PATH)

print("\nOUTPUT_DIR:")
print(OUTPUT_DIR)

print("\nNUM_THREADS:")
print(NUM_THREADS)

In [0]:
print("===================================================")
print("MACHINE CHECK")
print("===================================================")

vm = psutil.virtual_memory()

print(f"CPU logical cores       : {psutil.cpu_count(logical=True)}")
print(f"CPU physical cores      : {psutil.cpu_count(logical=False)}")
print(f"LightGBM NUM_THREADS    : {NUM_THREADS}")
print(f"Total memory GiB        : {vm.total / (1024 ** 3):,.2f}")
print(f"Available memory GiB    : {vm.available / (1024 ** 3):,.2f}")

print("===================================================")

In [0]:
# ============================================================
# Read original dataset
# ============================================================

import pyarrow.dataset as ds
import pyarrow as pa
import pandas as pd

dataset = ds.dataset(DATA_PATH, format="parquet")

# ============================================================
# Read SHAP selected features
# ============================================================

shap = pd.read_csv("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/final_selected_top_1000_features_by_shap.csv")
selected_features = shap["feature"].tolist()

selected_features = [
    c for c in selected_features
    if c in dataset.schema.names
]

# Add target column also
selected_features = selected_features + [TARGET_COL]

feature_cols = selected_features

schema = dataset.schema
all_cols = schema.names

if TARGET_COL not in all_cols:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

# ============================================================
# Helper: detect numeric Arrow types
# ============================================================

def is_arrow_numeric_type(dtype):
    return (
        pa.types.is_integer(dtype)
        or pa.types.is_floating(dtype)
        or pa.types.is_decimal(dtype)
        or pa.types.is_boolean(dtype)
    )

# ============================================================
# Select numeric feature columns
# ============================================================

feature_cols = []

for col_name in selected_features:
    field = schema.field(col_name)
    dtype = field.type

    if is_arrow_numeric_type(dtype):
        feature_cols.append(col_name)

n_features = len(feature_cols)

if n_features == 0:
    raise ValueError("No numeric/decimal feature columns found.")

print("Total columns in source dataset :", len(all_cols))
print("Numeric feature columns used    :", n_features)
print("Target column                   :", TARGET_COL)

print("Sample features:")
print(feature_cols[:20])

**SELECTING TOP 1000 FEATURES BASED ON SHAP**

In [0]:
cols = len(dataset.schema.names)

rows = dataset.count_rows()

print("Shape:", (rows, cols))

In [0]:
feature_aliases = [f"f_{i}" for i in range(n_features)]

feature_mapping_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols
})

# feature_mapping_path = os.path.join(OUTPUT_DIR, "feature_alias_mapping.csv")

# feature_mapping_df.to_csv(feature_mapping_path, index=False)

# print("Feature mapping saved to:")
# print(feature_mapping_path)

feature_mapping_df.head()

In [0]:
# ============================================================
# Count rows
# ============================================================

n_rows = dataset.count_rows()

print("Total rows:", n_rows)
print("Total features:", n_features)

# ============================================================
# Random train/test mask
# ============================================================

rng = np.random.default_rng(SEED)

train_mask_all = rng.random(n_rows) < TRAIN_RATIO

n_train = int(train_mask_all.sum())
n_test = int(n_rows - n_train)

print("Train rows:", n_train)
print("Test rows :", n_test)
print("Train ratio actual:", n_train / n_rows)

In [0]:
print("Allocating train/test arrays...")

start_time = time.time()

X_train = np.empty(
    shape=(n_train, n_features),
    dtype=ARRAY_DTYPE,
    order="C"
)

X_test = np.empty(
    shape=(n_test, n_features),
    dtype=ARRAY_DTYPE,
    order="C"
)

y_train = np.empty(
    shape=(n_train,),
    dtype=ARRAY_DTYPE
)

y_test = np.empty(
    shape=(n_test,),
    dtype=ARRAY_DTYPE
)

elapsed = time.time() - start_time

print(f"Allocation complete in {elapsed:,.2f} seconds")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

In [0]:
# ============================================================
# Columns to scan
# ============================================================

scan_cols = [TARGET_COL] + feature_cols

# Cast every scanned column to float32
cast_schema = pa.schema([
    (col, pa.float32())
    for col in scan_cols
])

# ============================================================
# Scanner
# ============================================================

scanner = dataset.scanner(
    columns=scan_cols,
    batch_size=ARROW_BATCH_SIZE,
    use_threads=True
)

# ============================================================
# Fill arrays batch by batch
# ============================================================

global_row_pos = 0
train_write_pos = 0
test_write_pos = 0

start_time = time.time()

for batch_id, record_batch in enumerate(scanner.to_batches(), start=1):
    batch_n = record_batch.num_rows

    # Convert this batch to Arrow table and cast decimals/numerics to float32
    table = pa.Table.from_batches([record_batch])
    table = table.cast(cast_schema, safe=False)

    # Convert batch to pandas, then NumPy
    # This is temporary and only batch-sized.
    pdf_batch = table.to_pandas(
        split_blocks=True,
        self_destruct=True
    )

    batch_arr = pdf_batch.to_numpy(
        dtype=np.float32,
        copy=True
    )

    # First column is target
    y_batch = batch_arr[:, 0]

    # Replace NULL / NaN / inf target with 0
    bad_y_mask = ~np.isfinite(y_batch)
    if bad_y_mask.any():
        y_batch[bad_y_mask] = 0.0

    # Remaining columns are features
    X_batch = batch_arr[:, 1:]

    # Replace feature inf with NaN.
    # Keep NaN as missing values for LightGBM.
    bad_x_mask = np.isinf(X_batch)
    if bad_x_mask.any():
        X_batch[bad_x_mask] = np.nan

    # Train/test split for this batch
    batch_train_mask = train_mask_all[
        global_row_pos : global_row_pos + batch_n
    ]

    batch_test_mask = ~batch_train_mask

    n_train_batch = int(batch_train_mask.sum())
    n_test_batch = int(batch_test_mask.sum())

    if n_train_batch > 0:
        X_train[
            train_write_pos : train_write_pos + n_train_batch,
            :
        ] = X_batch[batch_train_mask, :]

        y_train[
            train_write_pos : train_write_pos + n_train_batch
        ] = y_batch[batch_train_mask]

        train_write_pos += n_train_batch

    if n_test_batch > 0:
        X_test[
            test_write_pos : test_write_pos + n_test_batch,
            :
        ] = X_batch[batch_test_mask, :]

        y_test[
            test_write_pos : test_write_pos + n_test_batch
        ] = y_batch[batch_test_mask]

        test_write_pos += n_test_batch

    global_row_pos += batch_n

    if batch_id % 25 == 0:
        elapsed = time.time() - start_time
        print(
            f"Batch {batch_id} | rows loaded: {global_row_pos:,} / {n_rows:,} "
            f"| elapsed sec: {elapsed:,.1f}"
        )

    del pdf_batch, batch_arr, X_batch, y_batch, table
    gc.collect()

elapsed = time.time() - start_time

print("===================================================")
print("LOAD COMPLETE")
print("===================================================")
print(f"Rows loaded           : {global_row_pos:,}")
print(f"Train rows written    : {train_write_pos:,}")
print(f"Test rows written     : {test_write_pos:,}")
print(f"Elapsed seconds       : {elapsed:,.2f}")
print("===================================================")

if train_write_pos != n_train:
    raise ValueError(f"Train row mismatch: expected {n_train}, got {train_write_pos}")

if test_write_pos != n_test:
    raise ValueError(f"Test row mismatch: expected {n_test}, got {test_write_pos}")

In [0]:
def summarize_target(y, name):
    total = len(y)
    zero_count = int(np.sum(y == 0))
    nan_count = int(np.sum(~np.isfinite(y)))

    print(f"========== {name} ==========")
    print(f"Rows       : {total:,}")
    print(f"Zero count : {zero_count:,}")
    print(f"NaN/inf    : {nan_count:,}")
    print(f"Min        : {np.nanmin(y):,.6f}")
    print(f"Max        : {np.nanmax(y):,.6f}")
    print(f"Mean       : {np.nanmean(y):,.6f}")
    print()

summarize_target(y_train, "TRAIN TARGET")
summarize_target(y_test, "TEST TARGET")

In [0]:
# ============================================================
# Dataset params
# ============================================================

dataset_params = {
    "max_bin": MAX_BIN,
    "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,
    "feature_pre_filter": False
}

# ============================================================
# LightGBM Dataset objects
# ============================================================

train_data = lgb.Dataset(
    data=X_train,
    label=y_train,
    feature_name=feature_aliases,
    free_raw_data=False,
    params=dataset_params
)

test_data = lgb.Dataset(
    data=X_test,
    label=y_test,
    feature_name=feature_aliases,
    reference=train_data,
    free_raw_data=False,
    params=dataset_params
)

print("LightGBM Dataset objects created.")

In [0]:
# ============================================================
# LightGBM params
# ============================================================

lgb_params = {
    "objective": "regression_l2",
    "metric": ["rmse", "l1"],

    "boosting_type": "gbdt",

    "learning_rate": LEARNING_RATE,
    "num_leaves": NUM_LEAVES,
    "min_data_in_leaf": MIN_DATA_IN_LEAF,

    "feature_fraction": FEATURE_FRACTION,

    # No row sampling
    "bagging_fraction": BAGGING_FRACTION,
    "bagging_freq": BAGGING_FREQ,

    "max_bin": MAX_BIN,
    "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,

    "force_col_wise": True,

    "num_threads": NUM_THREADS,
    "seed": SEED,

    "verbosity": -1
}

# ============================================================
# Train
# ============================================================

start_time = time.time()

booster = lgb.train(
    params=lgb_params,
    train_set=train_data,
    num_boost_round=NUM_ITERATIONS,
    valid_sets=[train_data, test_data],
    valid_names=["train", "test"],
    callbacks=[
        lgb.log_evaluation(period=25)
    ]
)

elapsed = time.time() - start_time

print("===================================================")
print("TRAINING COMPLETE")
print("===================================================")
print(f"Elapsed seconds: {elapsed:,.2f}")
print("===================================================")

# model_path =  "lightgbm_nbrx_model.txt"

# booster.save_model(model_path)

# print("Model saved to:")
# print(model_path)

In [0]:
start_time = time.time()

pred_train = booster.predict(
    X_train,
    num_threads=NUM_THREADS
)

pred_test = booster.predict(
    X_test,
    num_threads=NUM_THREADS
)

elapsed = time.time() - start_time

print("Prediction complete.")
print(f"Elapsed seconds: {elapsed:,.2f}")

In [0]:
def calculate_metrics(y_true, y_pred, dataset_name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    valid_mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true_valid = y_true[valid_mask]
    y_pred_valid = y_pred[valid_mask]

    r2 = r2_score(y_true_valid, y_pred_valid)

    # MAPE: exclude rows where actual = 0
    nonzero_mask = y_true_valid != 0

    if nonzero_mask.sum() > 0:
        mape = (
            np.mean(
                np.abs(
                    (y_true_valid[nonzero_mask] - y_pred_valid[nonzero_mask])
                    / y_true_valid[nonzero_mask]
                )
            )
            * 100.0
        )
    else:
        mape = np.nan

    # WMAPE: sum absolute error / sum absolute actual
    wmape_denominator = np.sum(np.abs(y_true_valid))

    if wmape_denominator > 0:
        wmape = (
            np.sum(np.abs(y_true_valid - y_pred_valid))
            / wmape_denominator
        ) * 100.0
    else:
        wmape = np.nan

    result = {
        "dataset": dataset_name,
        "r2": float(r2),
        "mape_pct": float(mape) if np.isfinite(mape) else np.nan,
        "wmape_pct": float(wmape) if np.isfinite(wmape) else np.nan,
        "rows_total": int(len(y_true)),
        "rows_used_for_r2": int(valid_mask.sum()),
        "rows_used_for_mape": int(nonzero_mask.sum()),
        "rows_used_for_wmape": int(valid_mask.sum()),
        "zero_target_rows_excluded_from_mape": int((y_true_valid == 0).sum())
    }

    return result


train_metrics = calculate_metrics(y_train, pred_train, "train")
test_metrics = calculate_metrics(y_test, pred_test, "test")

metrics_df = pd.DataFrame([train_metrics, test_metrics])

print(metrics_df)

In [0]:
import pandas as pd

# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)

# Show full column width
pd.set_option('display.max_colwidth', None)

# Show full dataframe width
pd.set_option('display.width', None)

# Example
metrics_df

In [0]:
# ============================================================
# Feature importance
# ============================================================

gain_importance = booster.feature_importance(
    importance_type="gain"
)

split_importance = booster.feature_importance(
    importance_type="split"
)

importance_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols,
    "gain_importance": gain_importance.astype(float),
    "split_importance": split_importance.astype(float)
})

importance_df = importance_df.sort_values(
    by="gain_importance",
    ascending=False
).reset_index(drop=True)

importance_df["gain_rank"] = np.arange(1, len(importance_df) + 1)

importance_path = os.path.join(OUTPUT_DIR, "all_feature_importance_gain.csv")
importance_df.to_csv(importance_path, index=False)

print("All feature importance saved to:")
print(importance_path)

importance_df.head(50)

In [0]:
top_k_effective = min(TOP_K, len(importance_df))

top_features_df = (
    importance_df
    .head(top_k_effective)
    .copy()
    .reset_index(drop=True)
)

top_feature_indices = top_features_df["feature_index"].to_numpy(dtype=np.int32)
top_feature_aliases = top_features_df["feature_alias"].tolist()
top_feature_names = top_features_df["feature"].tolist()

print("Top features selected:", len(top_features_df))
print("First 20 selected features:")
print(top_feature_names[:20])

top_features_path = os.path.join(OUTPUT_DIR, "top_1000_features_gain_only.csv")
top_features_df.to_csv(top_features_path, index=False)

print("Top 1000 gain-only file saved to:")
print(top_features_path)

In [0]:
def aggregate_shap_for_top_features(
    booster,
    X,
    top_indices,
    batch_size,
    num_threads
):
    n = X.shape[0]
    k = len(top_indices)

    shap_sum = np.zeros(k, dtype=np.float64)
    shap_abs_sum = np.zeros(k, dtype=np.float64)

    rows_processed = 0
    start_time = time.time()

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)

        X_batch = X[start:end, :]

        # LightGBM returns shape:
        # rows x (num_features + 1)
        # Last column is expected value / base value.
        shap_batch_all = booster.predict(
            X_batch,
            pred_contrib=True,
            num_threads=num_threads
        )

        shap_batch_top = shap_batch_all[:, top_indices]

        shap_sum += shap_batch_top.sum(axis=0)
        shap_abs_sum += np.abs(shap_batch_top).sum(axis=0)

        rows_processed += end - start

        if rows_processed % (batch_size * 20) == 0 or rows_processed == n:
            elapsed = time.time() - start_time
            print(
                f"SHAP rows processed: {rows_processed:,} / {n:,} "
                f"| elapsed sec: {elapsed:,.1f}"
            )

        del X_batch, shap_batch_all, shap_batch_top
        gc.collect()

    mean_shap = shap_sum / rows_processed
    mean_abs_shap = shap_abs_sum / rows_processed

    return mean_shap, mean_abs_shap


start_time = time.time()

mean_shap_train, mean_abs_shap_train = aggregate_shap_for_top_features(
    booster=booster,
    X=X_train,
    top_indices=top_feature_indices,
    batch_size=SHAP_BATCH_SIZE,
    num_threads=NUM_THREADS
)

elapsed = time.time() - start_time

print("===================================================")
print("SHAP AGGREGATION COMPLETE")
print("===================================================")
print(f"Elapsed seconds: {elapsed:,.2f}")
print("===================================================")

In [0]:
top_features_with_shap_df = top_features_df.copy()

top_features_with_shap_df["mean_shap_train"] = mean_shap_train
top_features_with_shap_df["mean_abs_shap_train"] = mean_abs_shap_train

# Rank by SHAP also
top_features_with_shap_df["mean_abs_shap_rank_within_top_1000"] = (
    top_features_with_shap_df["mean_abs_shap_train"]
    .rank(method="first", ascending=False)
    .astype(int)
)

top_features_with_shap_path = os.path.join(
    OUTPUT_DIR,
    "top_1000_features_with_train_shap.csv"
)

top_features_with_shap_df.to_csv(
    top_features_with_shap_path,
    index=False
)

print("Top 1000 features with SHAP saved to:")
print(top_features_with_shap_path)

top_features_with_shap_df.head(50)

In [0]:
top_features_by_shap_df = (
    top_features_with_shap_df
    .sort_values("mean_abs_shap_train", ascending=False)
    .reset_index(drop=True)
)

top_features_by_shap_path = os.path.join(
    OUTPUT_DIR,
    "top_1000_features_ranked_by_mean_abs_shap.csv"
)

top_features_by_shap_df.to_csv(
    top_features_by_shap_path,
    index=False
)

print("Top 1000 features ranked by SHAP saved to:")
print(top_features_by_shap_path)

top_features_by_shap_df.head(50)

In [0]:
selected_feature_list_path = os.path.join(
    OUTPUT_DIR,
    "selected_top_1000_feature_names.txt"
)

with open(selected_feature_list_path, "w") as f:
    for col in top_feature_names:
        f.write(col + "\n")

print("Selected feature list saved to:")
print(selected_feature_list_path)